In [ ]:
import rebound
import reboundx
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
rebound.__version__

In [ ]:
date='2024-01-01 00:00'
sim = rebound.Simulation()
sim.add('Sun', date=date, hash='sun')
sim.add('Mercury', date=date)
sim.add('Venus', date=date)
sim.add('Earth', date=date, hash='earth')
sim.add('Mars', date=date)
sim.add('Jupiter', date=date)
sim.add('Saturn', date=date)
sim.add('Uranus', date=date)
sim.add('Neptune', date=date)
sim.move_to_com()
sim.convert_particle_units('AU', 'year', 'Msun')
# sim.save_to_file('ss.bin')
ps = sim.particles

In [ ]:
%%time
tmax = -3.5*1e9
Nsnaps = 100000
interval = int(abs(tmax/Nsnaps))

# sim= rebound.Simulation('test_030325.bin')
sim.integrator = "WHCKL" 
sim.ri_whfast.safe_mode = False
sim.ri_whfast.corrector = 17
sim.ri_whfast.keep_unsynchronized=True
sim.dt = 4.062/365.25
# sim.save_to_file('test_030325.bin', interval=interval, delete_file=True)

rebx = reboundx.Extras(sim)
gr = rebx.load_force('gr_potential')
rebx.add_force(gr)
gr.params['c'] = 63240 # speed of light in AU/yr

cf = rebx.load_force("quadrupole")
rebx.add_force(cf)

earth_m = ps['earth'].m/1.0123000370338813
f = 0.8525
mu_eff = f*(1*0.0123000370338813*(earth_m)**2)/(1.0123000370338813*earth_m)
R = 0.0025696
ps['earth'].params["Rcentral"] = R
ps['earth'].params["mu_effcentral"] = mu_eff

sim.integrate(tmax)

In [ ]:
sa = rebound.Simulationarchive("test_030325.bin")
sim = sa[0]
ps = sim.particles

rebx = reboundx.Extras(sim)
gr = rebx.load_force('gr_potential')
rebx.add_force(gr)
gr.params['c'] = 63240 # speed of light in AU/yr

cf = rebx.load_force("quadrupole")
rebx.add_force(cf)
earth_m = ps['earth'].m/1.0123000370338813
f = 0.8525
mu_eff = f*(1*0.0123000370338813*(earth_m)**2)/(1.0123000370338813*earth_m)
R = 0.0025696
ps['earth'].params["Rcentral"] = R
ps['earth'].params["mu_effcentral"] = mu_eff

E0 = sim.energy() + rebx.gr_potential_potential(gr) + rebx.quad_force_potential()
Eerr, times = np.zeros(len(sa)), np.zeros(len(sa))

for i, sim in enumerate(sa):
    ps = sim.particles
    rebx = reboundx.Extras(sim)
    gr = rebx.load_force('gr_potential')
    rebx.add_force(gr)
    gr.params['c'] = 63240 # speed of light in AU/yr

    cf = rebx.load_force("quadrupole")
    rebx.add_force(cf)
    earth_m = ps['earth'].m/1.0123000370338813
    f = 0.8525
    mu_eff = f*(1*0.0123000370338813*(earth_m)**2)/(1.0123000370338813*earth_m)
    R = 0.0025696
    ps['earth'].params["Rcentral"] = R
    ps['earth'].params["mu_effcentral"] = mu_eff

    sim.integrator_synchronize()
    E = sim.energy() + rebx.gr_potential_potential(gr) + rebx.quad_force_potential()
    Eerr[i] = np.abs((E-E0)/E0)
    times[i] = sim.t

In [ ]:
fig, ax = plt.subplots()
ax.plot(times[:-1], Eerr[:-1], '.')
ax.set_xscale('log')
ax.set_yscale('log')

# Reproducibility

In [ ]:
t = sa[2].t
sim = sa[2]

# Have to readd forces and parameters
ps = sim.particles
rebx = reboundx.Extras(sim)
gr = rebx.load_force('gr_potential')
rebx.add_force(gr)
gr.params['c'] = 63240 # speed of light in AU/yr

cf = rebx.load_force("quadrupole")
rebx.add_force(cf)
earth_m = ps['earth'].m/1.0123000370338813
f = 0.8525
mu_eff = f*(1*0.0123000370338813*(earth_m)**2)/(1.0123000370338813*earth_m)
R = 0.0025696
ps['earth'].params["Rcentral"] = R
ps['earth'].params["mu_effcentral"] = mu_eff

sim.integrator_synchronize()
sim.particles[1].x

In [ ]:
sim = sa[1]

# Have to readd forces and parameters
ps = sim.particles
rebx = reboundx.Extras(sim)
gr = rebx.load_force('gr_potential')
rebx.add_force(gr)
gr.params['c'] = 63240 # speed of light in AU/yr

cf = rebx.load_force("quadrupole")
rebx.add_force(cf)
earth_m = ps['earth'].m/1.0123000370338813
f = 0.8525
mu_eff = f*(1*0.0123000370338813*(earth_m)**2)/(1.0123000370338813*earth_m)
R = 0.0025696
ps['earth'].params["Rcentral"] = R
ps['earth'].params["mu_effcentral"] = mu_eff

sim.integrate(t, exact_finish_time=0)
sim.integrator_synchronize()
sim.particles[1].x

In [ ]:
print("hello")